In [ ]:
import re # Regular expressions
import nltk # Tokenisation
from nltk.util import ngrams
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from transformers import pipeline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

dataset = '/content/drive/MyDrive/Colab Notebooks/test_medical.txt'
with open(dataset, 'r') as f:
        text_content = f.read()

text_sample = text_content[:8260]

# text sample for the first 6 cases
print(text_sample)

Mounted at /content/drive
Excision of limbal dermoids. We reviewed the clinical files of 10 patients who had undergone excision of unilateral epibulbar limbal dermoids. Preoperatively, all of the affected eyes had worse visual acuity (P less than .02) and more astigmatism (P less than .01) than the contralateral eyes. Postoperatively, every patient was cosmetically improved. Of the eight patients for whom both preoperative and postoperative visual acuity measurements had been obtained, in six it had changed minimally (less than or equal to 1 line), and in two it had improved (less than or equal to 2 lines). Surgical complications included persistent epithelial defects (40%) and peripheral corneal vascularization and opacity (70%). These complications do not outweigh the cosmetic and visual benefits of dermoid excision in selected patients. 
Bell's palsy. A diagnosis of exclusion. In cases of acute unilateral facial weakness, a careful and systematic evaluation is necessary to identify 

#PRE-PROCESSING: RegEx and Tokenization

In [ ]:
# Text Cleaning using RegEx

text_no_space = re.sub(r"\s+", " ", text_content)
cleaned_text = text_no_space.lower()

# Clean text sample

text_no_space = re.sub(r"\s+", " ", text_content)
clean_text_sample = text_sample.lower()
#print(cleaned_text)
#print(clean_text_sample)


In [ ]:
# Tokenization
nltk.download('punkt', quiet=True)

nltk.download('punkt_tab')

from nltk.tokenize import word_tokenize

word_tokens = word_tokenize(cleaned_text)

#print("Tokens:", word_tokens)


# Tokenization sample_text

word_tokens_sample = word_tokenize(clean_text_sample)

print("Tokens:", word_tokens_sample)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Tokens: ['excision', 'of', 'limbal', 'dermoids', '.', 'we', 'reviewed', 'the', 'clinical', 'files', 'of', '10', 'patients', 'who', 'had', 'undergone', 'excision', 'of', 'unilateral', 'epibulbar', 'limbal', 'dermoids', '.', 'preoperatively', ',', 'all', 'of', 'the', 'affected', 'eyes', 'had', 'worse', 'visual', 'acuity', '(', 'p', 'less', 'than', '.02', ')', 'and', 'more', 'astigmatism', '(', 'p', 'less', 'than', '.01', ')', 'than', 'the', 'contralateral', 'eyes', '.', 'postoperatively', ',', 'every', 'patient', 'was', 'cosmetically', 'improved', '.', 'of', 'the', 'eight', 'patients', 'for', 'whom', 'both', 'preoperative', 'and', 'postoperative', 'visual', 'acuity', 'measurements', 'had', 'been', 'obtained', ',', 'in', 'six', 'it', 'had', 'changed', 'minimally', '(', 'less', 'than', 'or', 'equal', 'to', '1', 'line', ')', ',', 'and', 'in', 'two', 'it', 'had', 'improved', '(', 'less', 'than', 'or', 'equal', 'to', '2', 'lines', ')', '.', 'surgical', 'complications', 'included', 'persistent

# Feature Engineering: N-Gram

In [ ]:
from nltk import ngrams

# Tokenize the sentence into words for both the full dataset and the sample text

tokens = nltk.word_tokenize(cleaned_text)
first_five_bigrams  = tokens[:6]
first_five_trigrams = tokens[:7]


# Generate bigrams
bigrams = list(ngrams(first_five_bigrams, 2))

# Generate trigrams
trigrams = list(ngrams(first_five_trigrams, 3))

#print("Bigrams List (first 5):", bigrams)
#print("Trigrams List (first 5):", trigrams)



# Tokenize the sentence into words for the sample text

tokens = nltk.word_tokenize(clean_text_sample)
first_twenty_bigrams  = tokens[:20]
first_twenty_trigrams = tokens[:21]


# Generate bigrams
bigrams = list(ngrams(first_twenty_bigrams, 2))

# Generate trigrams
trigrams = list(ngrams(first_twenty_trigrams, 3))

print("Bigrams List (first 20):", bigrams)
print("Trigrams List (first 20):", trigrams)

Bigrams List (first 20): [('excision', 'of'), ('of', 'limbal'), ('limbal', 'dermoids'), ('dermoids', '.'), ('.', 'we'), ('we', 'reviewed'), ('reviewed', 'the'), ('the', 'clinical'), ('clinical', 'files'), ('files', 'of'), ('of', '10'), ('10', 'patients'), ('patients', 'who'), ('who', 'had'), ('had', 'undergone'), ('undergone', 'excision'), ('excision', 'of'), ('of', 'unilateral'), ('unilateral', 'epibulbar')]
Trigrams List (first 20): [('excision', 'of', 'limbal'), ('of', 'limbal', 'dermoids'), ('limbal', 'dermoids', '.'), ('dermoids', '.', 'we'), ('.', 'we', 'reviewed'), ('we', 'reviewed', 'the'), ('reviewed', 'the', 'clinical'), ('the', 'clinical', 'files'), ('clinical', 'files', 'of'), ('files', 'of', '10'), ('of', '10', 'patients'), ('10', 'patients', 'who'), ('patients', 'who', 'had'), ('who', 'had', 'undergone'), ('had', 'undergone', 'excision'), ('undergone', 'excision', 'of'), ('excision', 'of', 'unilateral'), ('of', 'unilateral', 'epibulbar'), ('unilateral', 'epibulbar', 'limb

#Feature Engineering: Vector Semantics (Vectorization, convert string to ML for NLP)

In [ ]:
import gensim
from gensim.models import Word2Vec

model=Word2Vec(sentences=cleaned_text, vector_size=150, window=7, epochs=20)
model=Word2Vec(sentences=clean_text_sample, vector_size=150, window=7, epochs=20)

# Chunking Text

In [ ]:
# We chunked the text to make it less computationally expensive and the model cannot generate more than 1024 tokens. Each chunk of 450 words is then summarized

# MODELS

In [ ]:
def chunk_by_words(clean_text_sample, max_words=450):
    words = clean_text_sample.split()
    chunks = []
    for i in range(0, len(words), max_words):
        chunk = " ".join(words[i:i + max_words])
        chunks.append(chunk)
    return chunks

chunks = chunk_by_words(clean_text_sample, max_words=450)
print(f"Number of chunks: {len(chunks)}")

for i, c in enumerate(chunks, 1):
    print(f"\n--- CHUNK {i} (words: {len(c.split())}) ---")
    print(c[:400], "...")

Number of chunks: 3

--- CHUNK 1 (words: 450) ---
excision of limbal dermoids. we reviewed the clinical files of 10 patients who had undergone excision of unilateral epibulbar limbal dermoids. preoperatively, all of the affected eyes had worse visual acuity (p less than .02) and more astigmatism (p less than .01) than the contralateral eyes. postoperatively, every patient was cosmetically improved. of the eight patients for whom both preoperative ...

--- CHUNK 2 (words: 450) ---
intrathecal morphine sulfate (itms) or placebo. after baseline ptscep, heart rate and, mean blood pressure were recorded, itms (15 micrograms.kg-1) was injected via standard dural puncture with the patient in the lateral position. ptsceps, heart rate, and mean blood pressure were recorded again at 5, 10, 20, 30, 60, 90, and 120 min. control patients were treated identically (including position, st ...

--- CHUNK 3 (words: 275) ---
research arrhythmia clinic, the duke population (154 patients). nine deaths occu

In [ ]:
# Using Hugging Face Transfomer

In [ ]:
# 1 SciFive

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "razent/SciFive-base-PubMed"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def summarize_chunk_scifive_causal(chunk, max_len=1024, min_len=40):
    prompt = (
        chunk
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_length=max_len,       # output summary length
            min_length=min_len,
            do_sample=False,
            num_beams=4,                # small beam search
            early_stopping=True,
            no_repeat_ngram_size=3,     # key line
            repetition_penalty=1.2,     # lightly penalize repeats
            eos_token_id=tokenizer.eos_token_id,
        )

    full = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    # try to strip the prompt if it appears
    if "Summary:" in full:
        full = full.split("Summary:")[-1].strip()
    return full

partial_summaries = []
for i, chunk in enumerate(chunks, 1):
    print(f"\n=== Summarizing chunk {i}/{len(chunks)} ===")
    s = summarize_chunk_scifive_causal(chunk, max_len=140, min_len=40)
    partial_summaries.append(s)
    print(s)

combined = " ".join(partial_summaries)

final_summary_scifive = summarize_chunk_scifive_causal(combined, max_len=1024, min_len=60)

print("\n" + "="*60)
print("FINAL SciFive SUMMARY")
print("="*60)
print(final_summary_scifive)


=== Summarizing chunk 1/3 ===
We performed a retrospective review of the medical records of patients who had undergone excision of epibulbar dermoids. We evaluated the cosmetic and visual outcomes after excision. We reviewed the results of 10 consecutive patients who underwent excisions of limbal dermoid lesions. In this study, we present our experience with the technique of excision in 10 patients. A retrospective study was performed to evaluate the clinical and visual results of the patients. a diagnosis of exclusion. idiopathic facial paralysis. ocular complications. - a case of explantation.

=== Summarizing chunk 2/3 ===
This study aimed to compare the effects of intrathecal morphine sulfate (itms) and placebo on ptscep waveforms during the awake state. 20 patients were randomly assigned to receive either itms or placebo. control patients received either saline or placebo in a double-blind, placebo-controlled study. 30 patients were randomized to receive one of the two treatments

In [ ]:
# 2 Pegasus

from transformers import AutoTokenizer, AutoModel, pipeline, PegasusTokenizer, PegasusForConditionalGeneration
import torch

# The Bio_ClinicalBERT model is an encoder-only model and not suitable for
# abstractive summarization with the transformers pipeline.
# To perform summarization, a sequence-to-sequence model is required.
# Let's use a Pegasus model for summarization as it's designed for this task.

# You can choose a specific Pegasus model, e.g., 'google/pegasus-cnn_dailymail'
summarization_model_name = "google/pegasus-cnn_dailymail"

summarizer_tokenizer = PegasusTokenizer.from_pretrained(summarization_model_name)
summarizer_model = PegasusForConditionalGeneration.from_pretrained(summarization_model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
summarizer_model.to(device)

# 3. Helper: summarize a single piece of text
def summarize_text(clean_text_sample, max_len=1024, min_len=40):
    inputs = summarizer_tokenizer(
        clean_text_sample,
        return_tensors="pt",
        truncation=True,
        max_length=1024,   # input limit
    ).to(device)

    with torch.no_grad():
        summary_ids = summarizer_model.generate(
            inputs["input_ids"],
            max_length=max_len,       # output summary length
            min_length=min_len,
            no_repeat_ngram_size=3,
            num_beams=4,                # small beam search
            early_stopping=True,
            repetition_penalty=1.2,
        )

    return summarizer_tokenizer.decode(summary_ids[0], skip_special_tokens=True)


# 4. Summarize each chunk, then summarize the summaries
partial_summaries = []
for i, chunk in enumerate(chunks, 1):
    print(f"\n=== Summarizing chunk {i}/{len(chunks)} ===")
    s = summarize_text(chunk, max_len=140, min_len=40)
    partial_summaries.append(s)
    print(s)

combined = " ".join(partial_summaries)
final_summary_pegasus = summarize_text(combined, max_len=1024, min_len=60)

print("\n" + "="*50)
print("FINAL SUMMARY")
print("="*50)
print(final_summary_pegasus)


Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== Summarizing chunk 1/3 ===
preoperatively, all of the affected eyes had worse visual acuity .<n> postoperatively, every patient was cosmetically improved.<n>intracranial fibromatosis.<n>Effect of intrathecal morphine on somatosensory evoked potentials in awake humans .

=== Summarizing chunk 2/3 ===
Itms does not affect ptscep waveforms in the 35-90 ms latency range during the awake state .<n> mortality in patients treated with flecainide and encainide for supraventricular arrhythmias .

=== Summarizing chunk 3/3 ===
Flecainide and encainide were used in patients with supraventricular arrhythmias .<n>The 6-year survival functions of these 2 populations, estimated by the kaplan-meier technique, did not differ significantly .

FINAL SUMMARY
Flecainide and encainide were used in patients with supraventricular arrhythmias .<n>The 6-year survival functions of these 2 populations, estimated by the kaplan-meier technique, did not differ significantly . on the basis of the patient's age .


In [ ]:
# 3 BioGPT

from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "microsoft/BioGPT"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

prompt = clean_text_sample + "\n\nSummary:"

inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

with torch.no_grad():
    summary_ids = model.generate(
        **inputs,
        max_new_tokens=160,   # how long the *generated summary* can be
        do_sample=False,
        num_beams=4,                # small beam search
        early_stopping=True,
        no_repeat_ngram_size=3,     # key line
        repetition_penalty=1.2,
        eos_token_id=tokenizer.eos_token_id,
    )

summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

def summarize_chunk_with_biogpt(chunk, max_new_tokens=120):
    prompt = ( chunk
        + "\n\nSummary:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=4,
            eos_token_id=tokenizer.eos_token_id,
        )

    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    # strip the prompt part if it appears in the output
    if "Summary:" in full_text:
        full_text = full_text.split("Summary:")[-1].strip()
    return full_text

partial_summaries = []
for i, chunk in enumerate(chunks, 1):
    print(f"\n=== Summarizing chunk {i}/{len(chunks)} ===")
    s = summarize_chunk_with_biogpt(chunk, max_new_tokens=120)
    partial_summaries.append(s)
    print(s)

# Combine and summarize again for a final “Med-PaLM style” summary
combined_text = " ".join(partial_summaries)
final_summary_biogpt = summarize_chunk_with_biogpt(combined_text, max_new_tokens=160)

print("\n" + "=" * 50)
print("FINAL SUMMARY")
print("=" * 50)
print(final_summary_biogpt)


=== Summarizing chunk 1/3 ===
excision of limbal dermoids. we reviewed the clinical files of 10 patients who had undergone excision of unilateral epibulbar limbal dermoids. preoperatively, all of the affected eyes had worse visual acuity (p less than .02) and more astigmatism (p less than .01) than the contralateral eyes. postoperatively, every patient was cosmetically improved. of the eight patients for whom both preoperative and postoperative visual acuity measurements had been obtained, in six it had changed minimally (less than or equal to 1 line), and in two it had improved (less than or equal to 2 lines). surgical complications included persistent epithelial defects (40%) and peripheral corneal vascularization and opacity (70%). these complications do not outweigh the cosmetic and visual benefits of dermoid excision in selected patients. bell's palsy. a diagnosis of exclusion. in cases of acute unilateral facial weakness, a careful and systematic evaluation is necessary to ident

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, util
from nltk.tokenize import sent_tokenize

model = SentenceTransformer("emilyalsentzer/Bio_ClinicalBERT")

def extractive_clinicalbert_summary(clinical_text_sample, num_sentences=3):
    # 1) Split into sentences
    sentences = sent_tokenize(clinical_text_sample)

    # If no sentences, return empty string
    if len(sentences) == 0:
        return ""

    # If text has fewer sentences than requested, just return all of them
    num_sentences = min(num_sentences, len(sentences))

    # turn each sentence into a numerical vector (= embedding).
    embeddings = model.encode(sentences, convert_to_tensor=True)

    # Document embedding (mean of sentence embeddings) -> shape: (1, D)
    doc_embedding = embeddings.mean(dim=0, keepdim=True)

    # Cosine similarity each sentence vs document to rank the document (predict next word)
    scores = util.cos_sim(embeddings, doc_embedding)

    scores = scores.squeeze(1)

    top_k = torch.topk(scores, k=num_sentences)
    top_indices = top_k.indices.cpu().numpy()

    summary_sentences = [sentences[i] for i in sorted(top_indices)]

    return " ".join(summary_sentences)

extractive_summary = extractive_clinicalbert_summary(clean_text_sample, num_sentences=4)
print(extractive_summary)


oral and topical steroids were used to induce regression in an inflammatory, obstructing endobronchial polyp caused by a retained foreign body. symptoms of recurrent ulceration and mucosal tags are well-described oral manifestations of crohn's disease; however, in our patient recurrent facial abscesses, which required extraoral drainage, also developed. accordingly, the influence of intrathecally administered morphine on posterior tibial nerve somatosensory cortical evoked potentials (ptsceps) was investigated in 22 unpremedicated, awake, neurologically normal patients scheduled to undergo elective abdominal or pelvic procedures. the purpose of this study was to assess whether an increased mortality risk also accompanied the use of these drugs to treat patients with supraventricular arrhythmias.


In [ ]:
!pip install sentence-transformers

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 33.1 MB/s eta 0:00:00


In [ ]:
! pip install sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 10.2 MB/s eta 0:00:00


# EVALUATION METRICS

In [ ]:
!pip install rouge-score
!pip install sacrebleu
!pip install bert-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=130c1d3cadc771d9597a8b176d557bf5b0f83918e51283709092939ec8e8793d
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 4.9 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement alignscore (from versions: none)
ERROR: No matching distribution found for alignscore
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.7 MB/s eta 0:00:00


In [ ]:
reference_summary = clean_text_sample   # gold summary from dataset
biogpt_summary = final_summary_biogpt
pegasus_summary = final_summary_pegasus
scifive_summary = final_summary_scifive
clinicalbert_summary = extractive_summary

# ROUGE

In [ ]:
from rouge_score import rouge_scorer

def eval_rouge(reference, prediction):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference, prediction)
    return scores

In [ ]:
print("ROUGE for BioGPT:", eval_rouge(reference_summary, biogpt_summary))
print("ROUGE for PEGASUS:", eval_rouge(reference_summary, pegasus_summary))
print("ROUGE for SciFive:", eval_rouge(reference_summary, scifive_summary))
print("ROUGE for ClinicalBERT:", eval_rouge(reference_summary, clinicalbert_summary))

ROUGE for BioGPT: {'rouge1': Score(precision=0.9927536231884058, recall=0.33827160493827163, fmeasure=0.5046040515653776), 'rouge2': Score(precision=0.9782082324455206, recall=0.33278418451400327, fmeasure=0.496619545175169), 'rougeL': Score(precision=0.9879227053140096, recall=0.3366255144032922, fmeasure=0.5021485573971762)}
ROUGE for PEGASUS: {'rouge1': Score(precision=0.9210526315789473, recall=0.02880658436213992, fmeasure=0.0558659217877095), 'rouge2': Score(precision=0.8108108108108109, recall=0.02471169686985173, fmeasure=0.04796163069544365), 'rougeL': Score(precision=0.8421052631578947, recall=0.02633744855967078, fmeasure=0.05107741420590583)}
ROUGE for SciFive: {'rouge1': Score(precision=0.828125, recall=0.043621399176954734, fmeasure=0.08287724784988272), 'rouge2': Score(precision=0.30158730158730157, recall=0.015650741350906095, fmeasure=0.029757243539545807), 'rougeL': Score(precision=0.421875, recall=0.022222222222222223, fmeasure=0.04222048475371384)}
ROUGE for Clinica

# BLEU

In [ ]:
import sacrebleu

def eval_bleu(reference, prediction):
    bleu = sacrebleu.corpus_bleu([prediction], [[reference]])
    return bleu.score

In [ ]:
print("BLEU for BioGPT:", eval_bleu(reference_summary, biogpt_summary))
print("BLEU for PEGASUS:", eval_bleu(reference_summary, pegasus_summary))
print("BLEU for SciFive:", eval_bleu(reference_summary, scifive_summary))
print("BLEU for ClinicalBERT:", eval_bleu(reference_summary, clinicalbert_summary))

BLEU for BioGPT: 14.188616799471504
BLEU for PEGASUS: 4.097724966852758e-12
BLEU for SciFive: 6.338764868010858e-06
BLEU for ClinicalBERT: 0.0028971967601255136


# BERT-SCORE

In [ ]:
from bert_score import score

def bert_score_eval(reference_summary, generated_summary, lang="en"):
    """
    Computes BERTScore Precision, Recall, and F1.
    Input: reference summary (string), generated summary (string)
    Output: dict with P, R, F1
    """
    P, R, F1 = score([generated_summary], [reference_summary], lang=lang)
    return {
        "BERTScore_P": float(P[0]),
        "BERTScore_R": float(R[0]),
        "BERTScore_F1": float(F1[0])
    }


print("BERTScore for BioGPT")
print(bert_score_eval(reference_summary, biogpt_summary))

print("\n BERTScore for PEGASUS")
print(bert_score_eval(reference_summary, pegasus_summary))

print("\n BERTScore for SciFive ")
print(bert_score_eval(reference_summary, scifive_summary))

print("\n BERTScore for ClinicalBERT (extractive)")
print(bert_score_eval(reference_summary, clinicalbert_summary))

BERTScore for BioGPT


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

{'BERTScore_P': 0.8038082718849182, 'BERTScore_R': 0.7672898173332214, 'BERTScore_F1': 0.7851245999336243}

 BERTScore for SciFive 


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'BERTScore_P': 0.8712948560714722, 'BERTScore_R': 0.7907458543777466, 'BERTScore_F1': 0.8290684819221497}

 BERTScore for ClinicalBERT (extractive)


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'BERTScore_P': 0.8557856678962708, 'BERTScore_R': 0.8117960691452026, 'BERTScore_F1': 0.8332105875015259}
